## Building Order Line Tables (Prior + Train)

### Purpose

We create two transaction-level datasets:

1. `silver_order_lines_prior`
   - Historical behavior
   - Used for feature engineering and segmentation

2. `silver_order_lines_train`
   - Used for supervised learning (target = reordered)

---

### Approach

We will:

1. Load Silver tables
2. Optimize joins
3. Build PRIOR dataset
4. Build TRAIN dataset
5. Validate outputs

In [27]:
orders = spark.table("silver_orders")
order_products_prior = spark.table("silver_order_products_prior")
order_products_train = spark.table("silver_order_products_train")
products = spark.table("silver_products")

print("Orders:", orders.count())
print("Prior:", order_products_prior.count())
print("Train:", order_products_train.count())

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 29, Finished, Available, Finished, False)

Orders: 3421083
Prior: 32434489
Train: 1384617


## Inspect Input Data

We verify schemas and sample rows before joining.

In [28]:
orders.show(5)
order_products_prior.show(5)
order_products_train.show(5)
products.show(5)

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 30, Finished, Available, Finished, False)

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|                8|                  NULL|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
+--------+-------+--------+------------+---------+-----------------+----------------------+
only showing top 5 rows

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+--------

### Filter by eval_set

In [29]:
orders_prior = orders.filter("eval_set = 'prior'")
orders_train = orders.filter("eval_set = 'train'")

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 31, Finished, Available, Finished, False)

In [30]:
# sanity check
orders_prior.select("eval_set").distinct().show()
orders_train.select("eval_set").distinct().show()

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 32, Finished, Available, Finished, False)

+--------+
|eval_set|
+--------+
|   prior|
+--------+

+--------+
|eval_set|
+--------+
|   train|
+--------+



## Join Optimization

- Repartition large tables by `order_id`
- Broadcast product table (small dimension)

This reduces shuffle and improves performance.

In [31]:
# after filtering
orders_prior = orders_prior.repartition("order_id")
orders_train = orders_train.repartition("order_id")

order_products_prior = order_products_prior.repartition("order_id")
order_products_train = order_products_train.repartition("order_id")

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 33, Finished, Available, Finished, False)

#### Broadcast Products

In [32]:
from pyspark.sql.functions import broadcast

products_broadcast = broadcast(products)

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 34, Finished, Available, Finished, False)

## Building `silver_order_lines_prior`

This dataset represents:

> Historical customer behavior

Used for:
- feature engineering
- clustering
- basket analysis

In [33]:
# join prior
silver_order_lines_prior = (
    order_products_prior
    .join(orders_prior, "order_id", "inner")
    .join(products_broadcast, "product_id", "left")
)

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 35, Finished, Available, Finished, False)

In [34]:
# select columns
silver_order_lines_prior = silver_order_lines_prior.select(
    "order_id",
    "user_id",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "product_id",
    "product_name",
    "aisle",
    "department",
    "add_to_cart_order",
    "reordered"
)

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 36, Finished, Available, Finished, False)

In [35]:
# validate prior
silver_order_lines_prior.show(5)
print("Prior rows:", silver_order_lines_prior.count())

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 37, Finished, Available, Finished, False)

+--------+-------+------------+---------+-----------------+----------------------+----------+--------------------+----------------+------------+-----------------+---------+
|order_id|user_id|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_id|        product_name|           aisle|  department|add_to_cart_order|reordered|
+--------+-------+------------+---------+-----------------+----------------------+----------+--------------------+----------------+------------+-----------------+---------+
|      26| 153404|           2|        0|               16|                   7.0|     35951|Organic Unsweeten...| soy lactosefree|  dairy eggs|                1|        0|
|      26| 153404|           2|        0|               16|                   7.0|     24852|              Banana|    fresh fruits|     produce|                2|        1|
|      26| 153404|           2|        0|               16|                   7.0|     46206|      Red Grapefruit|    fresh fruits|    

## Building `silver_order_lines_train`

This dataset is used for:

> Supervised learning

Target variable:
- `reordered` (classification)

In [36]:
# join train
silver_order_lines_train = (
    order_products_train
    .join(orders_train, "order_id", "inner")
    .join(products_broadcast, "product_id", "left")
)


StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 38, Finished, Available, Finished, False)

In [37]:
# select columns
silver_order_lines_train = silver_order_lines_train.select(
    "order_id",
    "user_id",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "product_id",
    "product_name",
    "aisle",
    "department",
    "add_to_cart_order",
    "reordered"
)

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 39, Finished, Available, Finished, False)

In [38]:
# validate train
silver_order_lines_train.show(5)
print("Train rows:", silver_order_lines_train.count())

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 40, Finished, Available, Finished, False)

+--------+-------+------------+---------+-----------------+----------------------+----------+--------------------+----------------+----------+-----------------+---------+
|order_id|user_id|order_number|order_dow|order_hour_of_day|days_since_prior_order|product_id|        product_name|           aisle|department|add_to_cart_order|reordered|
+--------+-------+------------+---------+-----------------+----------------------+----------+--------------------+----------------+----------+-----------------+---------+
|    1342| 156818|          32|        3|                8|                  30.0|     13176|Bag of Organic Ba...|    fresh fruits|   produce|                1|        1|
|    1342| 156818|          32|        3|                8|                  30.0|     30827|  Seedless Cucumbers|packaged produce|   produce|                2|        1|
|    1342| 156818|          32|        3|                8|                  30.0|     14966|   Organic Mandarins|    fresh fruits|   produce|   

## Writing Silver Tables

We persist both datasets for downstream use:

- PRIOR → feature engineering
- TRAIN → supervised models

In [39]:
# save
silver_order_lines_prior.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_order_lines_prior")

silver_order_lines_train.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_order_lines_train")

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 41, Finished, Available, Finished, False)

In [40]:
# final check
print("Prior:", spark.table("silver_order_lines_prior").count())
print("Train:", spark.table("silver_order_lines_train").count())

StatementMeta(, 0cfa11da-bcac-482f-afc1-04b5cf3540b6, 42, Finished, Available, Finished, False)

Prior: 32434489
Train: 1384617
